# Rank Prediction Baseline

This reference notebook loads official CSV game records, extracts features, trains a model and generates `submission_rank.csv`. The official release defines the authoritative dataset and submission format. The architecture, features and training approach may be adapted.

## Task
Given 5–20 games from one player, predict one of ten ordered ranks:
`D, C, B, A, 1D, 2D, 3D, 4D, 5D, 6D`.

## Environment requirements
| Item | Reference requirement |
|---|---|
| GPU | 9 GB VRAM; approximately 8.6 GB training and 0.6 GB inference |
| Disk | Approximately 1.7 GB features; 101 MB per checkpoint |
| Packages | torch, webdataset, sgfmill, pandas, numpy, matplotlib |

A Linux GPU environment is recommended. Smaller training batches reduce memory: the original tutorial estimates about 4.5 GB at 128 and 2.5 GB at 64. Batch changes can affect convergence.

The notebook uses Linux `fork` multiprocessing with notebook-defined worker functions. On Windows/macOS, the original guidance suggests `NPROC = 1` or moving workers into an importable `.py` module. Platform compatibility still requires testing.

The paths in the configuration cell must match the dataset location.


In [ ]:
import glob
import os
import time
from multiprocessing import Pool, Process

import numpy as np
import pandas as pd
import torch
import torch.optim as optim
import webdataset as wds
from torch.utils.data import DataLoader

from utils import SGFParseRankPrediction
from network import GoRankResNet

print('torch', torch.__version__, '| cuda', torch.version.cuda,
      '| webdataset', wds.__version__)

In [ ]:
# Official data locations: preserve the released training/ and tests/ structure
TRAIN_DIR = './training'
TEST_DIR = './tests'
FEAT_DIR = './rank-features'
CKPT_DIR = './rank-trained-models'

# Ranks run from weakest to strongest; adjacency in scoring uses this order.
LEVELS = ['D', 'C', 'B', 'A', '1D', '2D', '3D', '4D', '5D', '6D']
RANK_TO_ID = {lv: i for i, lv in enumerate(LEVELS)}
ID_TO_RANK = {i: lv for lv, i in RANK_TO_ID.items()}

NPROC = 8              # Worker processes for feature extraction
PER_SHARD = 1000       # Samples per WebDataset shard
SEED = 42
VAL_SIZE = 100_000     # Validation subset size taken from the training pool

BATCH_SIZE = 256
LEARNING_RATE = 1e-3
NUM_EPOCHS = 5
NUM_CLASSES = len(LEVELS)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

os.makedirs(CKPT_DIR, exist_ok=True)
print('device', DEVICE)
RANK_TO_ID

The shared training pool contains ten rank CSV files, each with 100,000 games, for a total of one million games. Columns are `player_id, game_id, rank, color, sgf_content`.

Rank prediction uses `sgf_content` as input and `rank` as the target. The full dataset is shuffled with the configured seed, then split into validation and training records. Class balance in a random validation subset is approximate rather than guaranteed.


In [ ]:
df = pd.concat([pd.read_csv(p) for p in sorted(glob.glob(f'{TRAIN_DIR}/train_*.csv'))],
               ignore_index=True)
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
print('Total: %s games' % format(len(df), ','))
print('Rank distribution', df['rank'].value_counts().to_dict())

df_val, df_train = df.iloc[:VAL_SIZE], df.iloc[VAL_SIZE:]
print('Split: training %s / validation %s' % (format(len(df_train), ','), format(len(df_val), ',')))
df.head(3)

# 1. Feature Extraction

This is a one-time preprocessing stage. Once shards exist, later training runs can start at Section 2 after the imports and configuration have run.

`SGFParseRankPrediction` converts a game to a `(500, 19, 19)` tensor: the last 250 board states, each represented by separate black and white planes. Short games receive empty board states at the beginning. Features are stored as `uint8` and converted to `float32` for training.


In [ ]:
_parser = SGFParseRankPrediction()
_demo = df.iloc[0]['sgf_content']
_feat, _err = _parser.features_from_sgf_content(_demo)
print('Feature shape for one game:', _feat.shape, _feat.dtype)

### Saving WebDataset shards

Features are stored on disk and streamed during training rather than kept entirely in RAM.

**Compression:** `.tar.gz` shards compress binary masks. The original tutorial reports approximately 180 GB uncompressed versus 1.5 GB training and 163 MB validation when compressed. ShardWriter and the reader handle gzip based on the filename.

**Parallel extraction:** each worker owns its parser and writes separate shards.

**Unique keys:** the rank prefix is included in `__key__` because `game_id` is unique only within a rank CSV. Without the prefix, records from different rank files can collide.


In [ ]:
def extract_worker(df_part, pattern, wid):
    """Extract a partition into shards; each worker owns a parser and ShardWriter."""
    parser = SGFParseRankPrediction()
    written = bad = 0
    t0 = time.time()
    with wds.ShardWriter(pattern, maxcount=PER_SHARD, verbose=0) as sink:
        for _, row in df_part.iterrows():
            feat, err = parser.features_from_sgf_content(row['sgf_content'])
            if err is not None:
                bad += 1
                continue
            sink.write({
                '__key__': '%s_%s' % (row['rank'], row['game_id']),
                'features.npy': feat,
                'label.txt': str(row['rank']),
                'rank_idx.txt': str(RANK_TO_ID[row['rank']]),
                'game_name.txt': str(row['game_id']),
            })
            written += 1
            if written % 20000 == 0:
                print('  [w%d] %d/%d  %.0f min'
                      % (wid, written, len(df_part), (time.time() - t0) / 60),
                      flush=True)
    print('  [w%d] Wrote %d records (%d parse failures), %.1f min'
          % (wid, written, bad, (time.time() - t0) / 60), flush=True)


def extract_split(df_part, tag):
    """Process one split and verify that its output is nonempty."""
    outdir = f'{FEAT_DIR}/{tag}'
    os.makedirs(outdir, exist_ok=True)

    procs = []
    for i in range(NPROC):
        pat = '%s/%s-w%d-%%06d.tar.gz' % (outdir, tag, i)
        p = Process(target=extract_worker, args=(df_part.iloc[i::NPROC], pat, i))
        p.start()
        procs.append(p)
    for p in procs:
        p.join()
        if p.exitcode != 0:
            raise SystemExit('%s: Worker exited abnormally (exitcode %s)' % (tag, p.exitcode))

    shards = glob.glob(f'{outdir}/*.tar.gz')
    total = sum(os.path.getsize(s) for s in shards)
    avg = total / len(shards) if shards else 0
    print('%s: %d shards, %.2f GB, average %.1f KB/shard'
          % (tag, len(shards), total / 1e9, avg / 1024), flush=True)
    # Check mean output size to detect empty compressed files despite normal logs.
    # File counts alone cannot detect empty payloads (about 46 bytes after gzip).
    if avg < 10 * 1024:
        raise SystemExit('%s: average shard size only %.0f bytes; shards appear empty' % (tag, avg))

In [ ]:
# One-time extraction. Once shards exist, later training starts at the next section.
t0 = time.time()
extract_split(df_val, 'val')
extract_split(df_train, 'train')
print('Completed; total time %.1f min' % ((time.time() - t0) / 60))

# 2. Data Loader

If feature extraction is complete, training can resume here after imports and configuration.

Multiple workers can reduce gzip decompression bottlenecks. Iterable datasets require correct worker partitioning to avoid reading the same data repeatedly. The helper counts validation records before accepting the candidate worker count; a mismatch triggers a fallback to zero workers.


In [ ]:
def decode_sample(sample):
    return (torch.from_numpy(sample['features.npy']).float(),
            torch.tensor(int(sample['rank_idx.txt']), dtype=torch.long),
            sample['game_name.txt'])


def make_loader(shards, workers, shuffle):
    ds = wds.WebDataset(shards, shardshuffle=1 if shuffle else 0)
    if shuffle:
        ds = ds.shuffle(1000)
    ds = ds.decode().map(decode_sample).batched(BATCH_SIZE, partial=True)
    return DataLoader(ds, batch_size=None, num_workers=workers,
                      pin_memory=True, persistent_workers=workers > 0)


def pick_workers(val_shards, candidate=4):
    """Count validation records; accept multiple workers only if the count matches."""
    total = sum(len(y) for _x, y, _n in make_loader(val_shards, candidate, False))
    if total == VAL_SIZE:
        print('num_workers=%d passed validation (%d records)' % (candidate, total))
        return candidate
    print('num_workers=%d read %d records (expected %d); falling back to 0' % (candidate, total, VAL_SIZE))
    return 0

In [ ]:
train_shards = sorted(glob.glob(f'{FEAT_DIR}/train/*.tar.gz'))
val_shards = sorted(glob.glob(f'{FEAT_DIR}/val/*.tar.gz'))
print('Shards: train %d / val %d' % (len(train_shards), len(val_shards)))

workers = pick_workers(val_shards)
train_loader = make_loader(train_shards, workers, True)
val_loader = make_loader(val_shards, workers, False)

_x, _y, _n = next(iter(val_loader))
print('One batch:', _x.shape, _y[:8].tolist())

# 3. Model Training

`GoRankResNet` applies an input convolution, 20 residual blocks and a fully connected classifier. The architecture is defined in `network.py`.

The ten output logits correspond to the ten rank labels. The number of output classes must match the label mapping.


In [ ]:
model = GoRankResNet(num_classes=NUM_CLASSES).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
criterion = torch.nn.CrossEntropyLoss()

print('Parameter count: {:,}'.format(sum(p.numel() for p in model.parameters())))
print('batch %d | lr %s | epochs %d' % (BATCH_SIZE, LEARNING_RATE, NUM_EPOCHS))

In [ ]:
def validate(model, loader, criterion):
    model.eval()
    loss_sum, correct, total, batches = 0.0, 0, 0, 0
    with torch.no_grad():
        for inputs, labels, _ in loader:
            inputs = inputs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            outputs = model(inputs)
            loss_sum += criterion(outputs, labels).item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            batches += 1
    model.train()
    return (loss_sum / batches if batches else 0.0,
            100 * correct / total if total else 0.0)

In [ ]:
# Keep the kernel running during training.
history = []
model.train()
best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):
    t0 = time.time()
    total_loss, correct, total, steps = 0.0, 0, 0, 0

    for inputs, labels, _ in train_loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        steps += 1
        if steps % 500 == 0:
            print('  ep%d step %d  loss %.4f  acc %.2f%%  %.1f min'
                  % (epoch + 1, steps, total_loss / steps,
                     100 * correct / total, (time.time() - t0) / 60), flush=True)

    train_acc = 100 * correct / total if total else 0.0
    val_loss, val_acc = validate(model, val_loader, criterion)
    history.append({'epoch': epoch + 1, 'train_acc': train_acc, 'val_acc': val_acc})
    print('Epoch %d/%d  train acc %.2f%%  |  val loss %.4f acc %.2f%%  |  %.1f min'
          % (epoch + 1, NUM_EPOCHS, train_acc, val_loss, val_acc,
             (time.time() - t0) / 60), flush=True)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), f'{CKPT_DIR}/best_model.pth')
        print('  New best validation accuracy %.2f%%; saved best_model.pth' % best_val_acc)
    torch.save(model.state_dict(), f'{CKPT_DIR}/epoch_%d.pth' % (epoch + 1))

pd.DataFrame(history).to_csv(f'{CKPT_DIR}/history.csv', index=False)
print('Training completed; best validation accuracy %.2f%%' % best_val_acc)

### Interpreting the training curves

Validation runs after each epoch. Increasing training accuracy with worsening validation performance can indicate overfitting.

Early validation accuracy can lag training accuracy while BatchNorm running statistics settle. A low first-epoch result should be interpreted with the later trend, rather than automatically treated as overfitting.

Inference uses `best_model.pth`, selected by highest validation accuracy, rather than necessarily using the final epoch.


In [ ]:
import matplotlib.pyplot as plt

hist = pd.read_csv(f'{CKPT_DIR}/history.csv')
best = hist.loc[hist['val_acc'].idxmax()]

plt.figure(figsize=(9, 4))
plt.plot(hist['epoch'], hist['train_acc'], marker='o', ms=3, label='train acc')
plt.plot(hist['epoch'], hist['val_acc'], marker='o', ms=3, label='val acc')
plt.axvline(best['epoch'], ls='--', c='gray', lw=1)
plt.annotate('best %.2f%%' % best['val_acc'],
             xy=(best['epoch'], best['val_acc']),
             xytext=(best['epoch'] + 1.5, best['val_acc'] - 4),
             arrowprops=dict(arrowstyle='->', color='gray'))
plt.xlabel('epoch'); plt.ylabel('accuracy (%)')
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

# 4. Inference with Testing Data On-the-Fly

Each question contains multiple games from one player. The code applies softmax to each game's logits, averages the probabilities, then chooses the highest-scoring class.

**Both public and private questions must be predicted and combined into one 800-row submission.** Missing questions invalidate a submission. Preserve the `pub_` and `priv_` question prefixes.


In [ ]:
GPU_BATCH = 64
_infer_parser = SGFParseRankPrediction()


def sgf_to_features(sgf_content):
    """Top-level helper for multiprocessing.Pool."""
    feat, err = _infer_parser.features_from_sgf_content(sgf_content)
    return None if err is not None else feat


def predict(weights_path, out_path):
    net = GoRankResNet(num_classes=NUM_CLASSES)
    net.load_state_dict(torch.load(weights_path, map_location='cpu'))
    net.eval().to(DEVICE)

    rows = []
    with Pool(NPROC) as pool:
        # Predict both test splits and concatenate into one submission
        for split in ('public', 'private'):
            test = pd.read_csv(f'{TEST_DIR}/rank_prediction_test_{split}.csv')
            sgf_cols = [c for c in test.columns if c.startswith('sgf_')]
            t0 = time.time()

            for idx, r in test.iterrows():
                # Filter empty sgf_ fields beyond num_games
                sgfs = [r[c] for c in sgf_cols
                        if pd.notna(r[c]) and str(r[c]).strip()]
                feats = [f for f in pool.map(sgf_to_features, sgfs) if f is not None]
                if not feats:
                    rows.append([r['question_id'], LEVELS[0]])
                    continue

                x = torch.from_numpy(np.stack(feats)).float()
                probs = []
                with torch.no_grad():
                    for i in range(0, len(x), GPU_BATCH):
                        out = net(x[i:i + GPU_BATCH].to(DEVICE))
                        probs.append(out.softmax(1).cpu().numpy())

                avg = np.concatenate(probs).mean(0)      # Average probabilities across games in the same question
                rows.append([r['question_id'], LEVELS[int(avg.argmax())]])
                if (idx + 1) % 100 == 0:
                    print('  %s %d/%d  %.0f sec'
                          % (split, idx + 1, len(test), time.time() - t0), flush=True)
            print('  %s completed: %d questions' % (split, len(test)), flush=True)

    sub = pd.DataFrame(rows, columns=['question_id', 'pred_rank'])
    sub.to_csv(out_path, index=False)
    print('Wrote %s (%d rows)' % (out_path, len(sub)))
    print('Prediction distribution:', sub['pred_rank'].value_counts().to_dict())
    return sub

In [ ]:
submission = predict(f'{CKPT_DIR}/best_model.pth', './submission_rank.csv')
submission.head()

# End of the tutorial

The original reference run reported a public score of **0.4631**. This is historical reference evidence, not a result produced by this translated copy. Private scores are published after the competition.

| Component | Reference run |
|---|---|
| GPU | NVIDIA GeForce RTX 4090, 24 GB, driver 535.288.01 |
| CPU | AMD EPYC-Genoa, 8 cores |
| RAM | 31 GB |
| OS | Ubuntu 22.04.5 LTS |
| Software | Python 3.10.12, PyTorch 2.6.0+cu124, CUDA 12.4 |

This tutorial is a reference implementation. Architecture, features and training methods can be adapted according to the competition rules.

